In [1]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

from rocobench.envs.task_sweep import SweepTask
from rocobench.policy import PlannedPathPolicy

import importlib
import mediapy
# Initialize parser

# Import parser module directly to avoid OpenAI key check in prompting/__init__.py
parser_path = '/iris/u/riadoshi/teams/robot-collab/prompting/parser.py'
spec = importlib.util.spec_from_file_location("parser_module", parser_path)
parser_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parser_module)
LLMResponseParser = parser_module.LLMResponseParser


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
env = SweepTask(np_seed=10)
parser = LLMResponseParser(
    env=env,
    llm_output_mode="action",
    robot_agent_names={
        "ur5e_robotiq": "Alice",
        "panda": "Bob",
    },
    response_keywords=["NAME", "ACTION"],
    direct_waypoints=3,
)

In [3]:
def generate_action_string(alice_action, bob_action):
    """Generate properly formatted action string for parser."""
    return f"""EXECUTE
        NAME Alice ACTION {alice_action}
        NAME Bob ACTION {bob_action}
        """

action_str = generate_action_string("SWEEP green_cube", "WAIT")
    
# Parse the action
obs = env.reset()
success, reason, path_plans = parser.parse(obs, action_str)

# Execute each path plan
all_frames = []
for path_plan in path_plans:
    policy = PlannedPathPolicy(
        physics=env.physics,
        robots=env.get_sim_robots(),
        path_plan=path_plan,
        control_freq=5,
        graspable_object_names=env.get_graspable_objects(),
        allowed_collision_pairs=env.get_allowed_collision_pairs(),
        timeout=1e6,
        skip_smooth_path=False,
    )
    
    # Plan the motion
    plan_success, plan_reason = policy.plan(env)
    if not plan_success:
        print(f"Planning failed: {plan_reason}")
    
    # Execute the plan
    while not policy.plan_exhausted:
        action = policy.act(obs, env.physics)
        obs = env.step(action)
        all_frames.append(env.physics.render(camera_id='panda_wrist_cam', height=480, width=640))

In [ ]:
mediapy.show_video(all_frames, fps=5)